# Figure 4 — how diverse are the N-termini and ECLs

Two views of diversity. The PCA asks whether the segments occupy different physicochemical territory from each other; the clustered identity heatmaps show the raw similarity structure, including whether it is driven by a few tight families against an otherwise flat background.

`HEAT` picks which two segments get heatmaps.

## Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import style

style.apply()
if not style.available():
    print("Arial not found, matplotlib fell back to DejaVu Sans")

RESULTS, FIGS = ROOT / "results", ROOT / "figures"
FIGS.mkdir(exist_ok=True)
PAL = style.PALETTE
SEGMENTS = ["N-term", "ECL1", "ECL2", "ECL3"]

In [ ]:
def pid_matrix(seg):
    return pd.read_csv(RESULTS / f"pid_{seg}.csv.gz", index_col=0)


def pid_values(seg):
    v = pid_matrix(seg).values.astype(float)
    iu = np.triu_indices_from(v, k=1)
    return v[iu][~np.isnan(v[iu])]

In [ ]:
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import squareform


def pca(X):
    """Standardise, then SVD. Returns 2D scores and explained variance."""
    X = np.asarray(X, dtype=float)
    X = np.nan_to_num(X, nan=np.nanmean(X))
    X = (X - X.mean(0)) / np.where(X.std(0) == 0, 1, X.std(0))
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    return U[:, :2] * S[:2], (S ** 2 / (S ** 2).sum())[:2]

## Load

In [ ]:
seg = pd.read_csv(RESULTS / "segments_long.csv")
FEATS = ["length", "n_cys", "n_sequon", "charge_density", "gravy",
         "f_aromatic", "f_hydrophobic", "f_gly_pro"]
HEAT = ("N-term", "ECL2")

d = seg[seg.segment.isin(SEGMENTS)].dropna(subset=FEATS)
X = d[FEATS].copy()
X["length"] = np.log10(X["length"].clip(lower=1))
xy, ev = pca(X.values)
print(f"PC1 {ev[0]:.1%}, PC2 {ev[1]:.1%}")

## Panels

In [ ]:
fig, ax = plt.subplot_mosaic([["pca", "pca"], list(HEAT)],
                             figsize=(10.5, 9), constrained_layout=True)

a = ax["pca"]
for s in SEGMENTS:
    m = (d["segment"] == s).values
    a.scatter(xy[m, 0], xy[m, 1], s=10, alpha=0.55, color=PAL[s], label=s,
              edgecolors="none")
a.set_xlabel(f"PC1 ({ev[0]:.0%})")
a.set_ylabel(f"PC2 ({ev[1]:.0%})")
a.set_title("Physicochemical space (log length, charge, GRAVY, aromaticity, "
            "Cys, sequons, G/P)")
a.legend()

for s in HEAT:
    a = ax[s]
    M = np.nan_to_num(pid_matrix(s).values.astype(float), nan=0.0)
    M = (M + M.T) / 2
    np.fill_diagonal(M, 1.0)
    D = 1 - M
    np.fill_diagonal(D, 0.0)
    order = leaves_list(linkage(squareform(D, checks=False), method="average"))
    vmax = min(1.0, np.percentile(M[np.triu_indices_from(M, 1)], 99.5))
    im = a.imshow(M[np.ix_(order, order)], cmap="magma", vmin=0, vmax=vmax)
    a.set_xticks([]); a.set_yticks([])
    a.set_title(f"{s} identity ({M.shape[0]} receptors, clustered)")
    fig.colorbar(im, ax=a, fraction=0.046, label="identity")

fig.savefig(FIGS / "04_diversity.png")